###DataLake (Deltalake) + Lakehouse (Deltatables) - using Delta format (parquet+snappy+delta log)

%md
Delta Lake is an open-source storage framework that brings reliability, ACID transactions, and performance to data lakes. It sits on top of Parquet files and is most commonly used with Apache Spark and Databricks.<br>
Delta Lake & Deltalakhouse is the Core/Analytical storage layer behind Bronze–Silver–Gold (medallion) architectures.
<img src="https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-logo-whitebackground.png" style="width:300px; float: right"/>

## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) Creating our first Delta Lake table

Delta is the default file and table format using Databricks.

In [0]:
spark.sql(f"create catalog if not EXISTS lakehousecat")

In [0]:
spark.sql(f"create schema if not exists lakehousecat.deltadb")

In [0]:
spark.sql(f"create volume if not exists lakehousecat.deltadb.datalake")

%md
####1. Write data into delta file (Datalake) and table (Lakehouse)
1. How to migrate csv to delta format
2. Difference between Delta and Parquet
2. How to create Datalake & Lakehouse

In [0]:
df=spark.read.csv("/Volumes/lakehousecat/deltadb/datalake",header=True,inferSchema=True)
display(df)
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat/deltadb/datalake/deltadir")
df.write.format("parquet").mode("overwrite").save("/Volumes/lakehousecat/deltadb/datalake/parquetdir")


In [0]:
df.write.save("/Volumes/lakehousecat/deltadb/datalake/defaultdir",mode='overwrite')

In [0]:
df.write.saveAsTable("lakehousecat.deltadb.drugstbl",mode='overwrite')

In [0]:
%sql
explain select * from lakehousecat.deltadb.drugstbl

In [0]:
#We can add Schema evolution feature just by adding the below option in Delta tables.
df.write.option("mergeSchema","true").saveAsTable("lakehousecat.deltadb.drugstbl1",mode='overwrite')
df=spark.read.table("lakehousecat.deltadb.drugstbl1")


%md
####2. DML Operations in Delta Tables & Files
- We are overcoming the WORM (Write Once Read Many) limitation in Cloud S3/GCS/ADLS or in Distributed storage layers like HDFS
- Delta file/table supports WMRM(Write Manay Read Many) operations, using DMLs such as  INSERT/DELETE/UPDATE/MERGE

In [0]:
%sql
create or replace table lakehousecat.deltadb.sample(id int,name string)
using delta;
insert into lakehousecat.deltadb.sample values(1,'mohan')


In [0]:
%sql
desc history lakehousecat.deltadb.sample

In [0]:
%sql

use lakehousecat.deltadb

In [0]:
%sql

desc history lakehousecat.deltadb.drugstbl

In [0]:
%sql
--DQL data query language
select * from drugstbl where uniqueid=163740;

In [0]:
%sql
--DML
update drugstbl set rating=rating-1 where uniqueid=163740;
    

In [0]:
%sql
select * from drugstbl where uniqueid=163740;

In [0]:
%sql
---DML operation delete
delete from drugstbl where uniqueid=163740;

In [0]:
%sql
SELECT * FROM drugstbl
where uniqueid in (163740,206473);
    
--DML operation merge

In [0]:
%sql
desc history lakehousecat.deltadb.drugstbl

%md
#####c. File DML (Update/Delete)
We don't do file DML usually, we are doing here just for learning about 
- file also can be undergone with limited DML operation
- we need to learn about how the background delta operation is happening when i do DML

In [0]:
spark.read.format("delta").load("/Volumes/lakehousecat/deltadb/datalake/deltadir/").where('uniqueid=163740').show()

In [0]:
#DML on Files: How to update delta files (Not used very frequently)
from delta.tables import DeltaTable
deltafile = DeltaTable.forPath(spark, "/Volumes/lakehousecat/deltadb/datalake/deltadir")
deltafile.update("uniqueid=163740", { "rating": "rating - 1" } )

In [0]:
spark.read.format("delta").load("/Volumes/lakehousecat/deltadb/datalake/deltadir/").where('uniqueid=163740').show()

%md
#####d. File Delete

In [0]:
df=spark.read.format("delta").load("/Volumes/lakehousecat/deltadb/datalake/deltadir/").where('uniqueid=163740')
df.where('uniqueid=163740').show()

In [0]:
from delta.tables import DeltaTable
deltaTable = DeltaTable.forPath(spark, "/Volumes/lakehousecat/deltadb/datalake/deltadir/")
deltaTable.delete("uniqueid=163740")

In [0]:
df=spark.read.format("delta").load("/Volumes/lakehousecat/deltadb/datalake/deltadir/").where('uniqueid=163740')
df.where('uniqueid=163740').show()

#####d. Merge Operation

In [0]:
%sql

select count(1) from drugstbl

In [0]:
%sql
create or replace table drugstbl_merge as select * from drugstbl where rating<=8;


In [0]:
%sql

select count(1) from drugstbl_merge

In [0]:
%sql
---merge into target table using source table on contionto join
---when contion matched then update
---when contition not matched then insert
---when contition not matched with cource data then delete from target table exsesus data


--merge syntax
--merge into targetable using sourcetable on condition_to_join
--when matched then update
--when not matched then insert
--when not matched by source then delete (some additional data in the target should be deleted, which is already deleted in source)
--Delta table support merge operation for (insert/update/delete)
--2899 updated
--2801 inserted
MERGE INTO drugstbl_merge tgt--2899
USING drugstbl src--5700
ON tgt.uniqueid = src.uniqueid
WHEN MATCHED THEN--2899 update
  UPDATE SET tgt.usefulcount= src.usefulcount,
             tgt.drugname = src.drugname,
             tgt.condition = src.condition
WHEN NOT MATCHED--2801 insert
  THEN INSERT (uniqueid,rating,date,usefulcount, drugname, condition ) VALUES (uniqueid,rating,date,usefulcount, drugname, condition);

In [0]:
%sql
select count(*) from drugstbl_merge

In [0]:
%sql
--What if the source got some data removed and which is present in the target still (we can leave it or delete)
insert into drugstbl_merge select 99999999,drugname,condition,rating,date,usefulcount 
from drugstbl limit 1;

In [0]:
%sql
select count(*) from drugstbl_merge

In [0]:
%sql
select count(*) from drugstbl

In [0]:
%sql
--Delta table support merge operation for (delete)
--1 deleted (which is not present in the source (source system deleted it already, hence target also has to delete))
MERGE INTO drugstbl_merge tgt
USING drugstbl src
ON tgt.uniqueid = src.uniqueid
WHEN MATCHED THEN
  UPDATE SET tgt.usefulcount= src.usefulcount,
             tgt.drugname = src.drugname,
             tgt.condition = src.condition
WHEN NOT MATCHED
  THEN INSERT (uniqueid,rating,date,usefulcount, drugname, condition ) VALUES (uniqueid,rating,date,usefulcount, drugname, condition)
WHEN NOT MATCHED BY SOURCE THEN DELETE;

####3. Additional Operations on Deltalake & Deltatables

#####a. History & Versioning
*History* returns one row per commit/version and tells you what changed, when, and how.

In [0]:
%sql

desc history drugstbl_merge

In [0]:
%sql

select count(*) from drugstbl_merge version as of 4;

#####b. Time Travel
*Timestamp as of* Reads the table as it existed at that exact timestamp and Any commits after the given timestamp is ignored

In [0]:
%sql
desc history drugstbl_merge

In [0]:
%sql

select count(*) from drugstbl_merge timestamp as of '2026-02-03T11:11:20.000+00:00';

In [0]:
%sql
desc history catalog1_we47.schema1_we47.silver_staff

#####c. Vaccum
*VACUUM* in Delta Lake removes old, unused files to free up storage, default retention hours is 168. These files come from operations like DELETE, UPDATE, or MERGE and are kept temporarily so time-travel queries can work.<br>

Before VACUUM<br>
Active + deleted parquet files exist<br>

After VACUUM<br>
Only ACTIVE parquet files remains and delete Old parquet files (from UPDATE/MERGE/DELETE)<br>
Logs remain intact<br>
Time travel beyond retention becomes impossible<br>

In [0]:
%sql

desc history drugstbl_merge

In [0]:
%sql
 vacuum drugstbl_merge retain 168 hours

#####d. ACID Transactions
**Delta Lake supports ACID transactions under the hood via a transaction log.**
| ACID        | In Databricks         |
| ----------- | --------------------- |
| Atomicity   | Every transactions are Individual Transactions / All or nothing  |
| Consistency | Schema + constraints  |
| Isolation   | Using Version/Time/restore we can isolate transactions, we can't use TCL (commit/rollback) |
| Durability  | Every transaction Always hit the disk (durable), but can be controlled by Transaction log |

In [0]:
%sql

use lakehousecat.deltadb

In [0]:
%sql
CREATE OR REPLACE TABLE acid_demo_txn (
  id INT,
  amount INT
) USING DELTA;

In [0]:
%sql
select * from acid_demo_txn;

In [0]:
%sql
--All or nothing (Atomic/Individual Transaction) - help us make a transaction complete or fail, hence no partial data
INSERT INTO acid_demo_txn VALUES--One atomic (all or nothing) transaction is inserting 3 rows
(1, 100),
(2, 200),
(3, 300);
--(3, '300');
--This will make the entire transaction failed (all or nothing)

In [0]:
%sql
desc history acid_demo_txn

In [0]:
%sql
select * from acid_demo_txn

In [0]:
%sql
restore table acid_demo_txn to version as of 0

In [0]:
%sql
select * from acid_demo_txn